In [ ]:
# 导入依赖与绘图默认样式
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "figure.facecolor": "white",
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.2,
    "font.size": 10,
})
from MDAnalysis import transformations as trans
import nglview as nv
import warnings
warnings.filterwarnings("ignore")

from _helper_functions import *

## 轨迹可视化

读取 LAMMPS 拓扑/轨迹，用 nglview 交互查看体系结构。

In [ ]:
# --- 输入文件（当前目录下的模拟输出）---
data = './data.lmp'                 # 初始 data：用于解析 type → 元素映射
topo = './result_atoms.data'       # 拓扑（含原子类型、电荷、坐标）
traj = './result_atoms.lammpstrj'  # 轨迹 dump

# 从 data 的 Masses 注释中读取 type → 元素符号
type_to_element = get_type_element_map(data)
type_to_element

# 残基命名：前 2 个残基为墙，其余为水分子（需与体系残基数一致）
resnames = ['wall'] * 2 + ['H2O'] * 2000

# 读入轨迹（dt 单位：fs，与 dump 时间步一致）
u = Trajectory.read_lammps_dump(topo, traj, type_to_element, resnames, dt=1)

# 在线变换：把原点移到盒子角点，并按残基做周期性包裹
workflow = [
    move_origin_to_corner,
    trans.wrap(u.atoms, compound='residues'),
]
u.trajectory.add_transformations(*workflow)

# 构建 nglview 视图
view = nv.show_mdanalysis(u)
view.clear_representations()
view.add_representation(selection='H2O', repr_type='licorice', radius='.5', opacity=1)
view.add_representation(selection='wall', repr_type='ball+stick', radius='.5', opacity=1)
view.add_unitcell()
# view.camera = 'orthographic'
view.control.spin([1, 0, 0], -np.pi / 2.)  # 绕 x 轴旋转，便于侧视墙–流体–墙结构

In [ ]:
# 单独一格显示：便于前端正确挂载播放控件
view

In [ ]:
# 导出轨迹最后一帧为 PNG（写到当前目录）
# render_image 为异步渲染，需短暂等待后再写盘
from time import sleep
import base64

view.frame = view.max_frame
sleep(1.0)
view.render_image(factor=4, antialias=True, trim=True, transparent=True)
sleep(2.0)

with open("last_frame.png", "wb") as f:
    f.write(base64.b64decode(view._image_data))
print("saved: last_frame.png  (frame =", view.frame, "/", view.max_frame, ")")

## 热力学与结构分析

从 `log.lammps` 与 `result_profile_stress.log` 读取温度、上墙高度、密度与局部压强剖面。

In [ ]:
# 流体温度与上墙高度随时间变化（1×2 组图）
data = read_result_thermo("log.lammps")
t0 = data["time"].iloc[0]
x = (data["time"] - t0) / 1e6  # fs → ns

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8.5, 2.4))

ax0.plot(x, data["temp"], "-")
ax0.set_xlabel("time [ns]")
ax0.set_ylabel("temperature [K]")
ax0.set_xlim(0, None)
ax0.tick_params(direction="in")

ax1.plot(x, data["upper_cmz"], "-")  # Å；机械控压的活塞位置
ax1.set_xlabel("time [ns]")
ax1.set_ylabel("height [Å]")
ax1.set_xlim(0, None)
ax1.tick_params(direction="in")

fig.tight_layout()
fig.savefig("temp_wall_height.png", bbox_inches="tight")
plt.show()
print("saved: temp_wall_height.png")

In [ ]:
# 沿 z 的数密度与局部压强（LAMMPS ave/chunk → result_profile_stress.log）
names = ["id", "z", "n", "sxx", "syy", "szz", "den"]
stress = pd.read_csv(
    "result_profile_stress.log",
    header=None,
    names=names,
    sep=r"\s+",
    skiprows=4,
)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(8.5, 2.4))

x = stress["z"] / 10  # Å → nm

# 左图：数密度；虚线标出 ~33.3 nm^-3（SPC/E 常温常压体相密度参考）
ax0.plot(x, stress["den"] / 3 * 1e3, color="tab:blue")
ax0.plot((-10, 10), (33.3, 33.3), "k--", lw=1)
ax0.set_xlabel("z [nm]")
ax0.set_ylabel(r"density [nm$^{-3}$]")
ax0.set_xlim(-4, 4)
ax0.set_ylim(0, 50)
ax0.tick_params(direction="in")

# 右图：局部压强 P ≈ -(sxx+syy+szz)/3 * den
press = -(stress["sxx"] + stress["syy"] + stress["szz"]) / 3 * stress["den"]
ax1.plot(x, press, color="tab:blue")
ax1.set_xlabel("z [nm]")
ax1.set_ylabel("pressure [atm]")
ax1.set_xlim(-4, 4)
ax1.set_ylim(-1e3, 1e3)
ax1.tick_params(direction="in")

fig.tight_layout()
fig.savefig("density_pressure.png", bbox_inches="tight")
plt.show()
print("saved: density_pressure.png")